# Model selection — sanity check and hyperparameter tuning

One-off analysis: confirm LightGBM is worth its complexity over a plain linear regression on this data, then tune LightGBM's hyperparameters. See `docs/decisions/0004-feature-engineering-and-baselines.md` for why this project uses this two-step approach instead of a full multi-model bake-off, and for the exact train/validation windows used below.

In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_pinball_loss

from felec.modeling.features import build_features

df = pd.read_parquet("../data/processed/dataset.parquet")
df = df.set_index("date_heure").sort_index()
features = build_features(df)
target = df["consommation"]

# Split dates are the fixed windows from ADR 0004. The index is UTC, so these
# slices land on UTC calendar-day boundaries, not Paris-local midnight (~2h
# offset in CEST) -- harmless here since the split stays contiguous and causal,
# just not exactly Paris-local like cutoff_instant_for_date elsewhere.
fit_data = features.loc[:"2025-08-15"].dropna(subset=["lag_168h"])
fit_target = target.loc[fit_data.index].dropna()
fit_data = fit_data.loc[fit_target.index]

val_data = features.loc["2025-08-16":"2025-10-15"]
val_target = target.loc[val_data.index].dropna()
val_data = val_data.loc[val_target.index]

len(fit_data), len(val_data)

(13318, 1464)

## 1. Sanity check: does gradient boosting earn its complexity?

In [2]:
linreg = LinearRegression()
# LinearRegression cannot handle NaN; LightGBM handles it natively so this
# workaround is only needed here. 0 is a safe fallback in this window because
# the lag_168h dropna above already removes the only rows with NaNs -- not a
# general imputation strategy (0degC would be a real value, not "missing").
linreg.fit(fit_data.fillna(0), fit_target)
linreg_pred = linreg.predict(val_data.fillna(0))
linreg_mae = mean_absolute_error(val_target, linreg_pred)

lgbm_default = lgb.LGBMRegressor(random_state=0, verbosity=-1)
lgbm_default.fit(fit_data, fit_target)
lgbm_pred = lgbm_default.predict(val_data)
lgbm_mae = mean_absolute_error(val_target, lgbm_pred)

print(f"Linear regression -- MAE: {linreg_mae:.0f} MW")
print(f"LightGBM (default) -- MAE: {lgbm_mae:.0f} MW")

Linear regression -- MAE: 1841 MW
LightGBM (default) -- MAE: 723 MW


## 2. Hyperparameter tuning (pinball loss, alpha=0.5)

In [3]:
param_grid = [
    {"num_leaves": 15, "learning_rate": 0.05, "n_estimators": 200, "min_child_samples": 20},
    {"num_leaves": 31, "learning_rate": 0.05, "n_estimators": 200, "min_child_samples": 20},
    {"num_leaves": 31, "learning_rate": 0.1, "n_estimators": 100, "min_child_samples": 20},
    {"num_leaves": 63, "learning_rate": 0.05, "n_estimators": 300, "min_child_samples": 10},
]

results = []
for params in param_grid:
    model = lgb.LGBMRegressor(
        objective="quantile", alpha=0.5, random_state=0, verbosity=-1, **params
    )
    model.fit(fit_data, fit_target)
    pred = model.predict(val_data)
    loss = mean_pinball_loss(val_target, pred, alpha=0.5)
    results.append({**params, "pinball_loss": loss})

results_df = pd.DataFrame(results).sort_values("pinball_loss")
results_df

,num_leaves,learning_rate,n_estimators,min_child_samples,pinball_loss
3,63,0.05,300,10,345.706934
1,31,0.05,200,20,361.903771
2,31,0.10,100,20,365.455997
0,15,0.05,200,20,398.677538


The winning combination becomes `DEFAULT_LGBM_PARAMS` in `src/felec/modeling/backtest.py` (Task 5), held fixed for every retrain in the walk-forward backtest.